In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType, TimestampType
import pandas as pd

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 21:21:41 WARN Utils: Your hostname, minh-HP-250-G8-Notebook-PC, resolves to a loopback address: 127.0.1.1; using 192.168.1.12 instead (on interface wlo1)
26/03/09 21:21:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 21:21:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 21:21:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
df_casualty = spark.read \
    .option("header", "true") \
    .csv('data/dft-road-casualty-statistics-casualty-1979-latest-published-year.csv')

In [5]:
df_casualty.head()

Row(collision_index='197905H200700', collision_year='1979', collision_ref_no='05H200700', vehicle_reference='1', casualty_reference='1', casualty_class='1', sex_of_casualty='1', age_of_casualty='20', age_band_of_casualty='4', casualty_severity='3', pedestrian_location='0', pedestrian_movement='0', car_passenger='0', bus_or_coach_passenger='0', pedestrian_road_maintenance_worker='-1', casualty_type='109', casualty_imd_decile='-1', lsoa_of_casualty='-1', enhanced_casualty_severity='-1', casualty_injury_based='-1', casualty_adjusted_severity_serious=None, casualty_adjusted_severity_slight=None, casualty_distance_banding='-1')

In [6]:
columns_casualty = [
    'collision_index',
    'casualty_class',
    'sex_of_casualty',
    'age_of_casualty',
    'casualty_severity',
    'casualty_type',
]

In [7]:
col_to_drop = []
for col in df_casualty.columns:
    if col not in columns_casualty:
        col_to_drop.append(col)

In [8]:
df_casu_drop_col = df_casualty.drop(*col_to_drop)

In [9]:
df_casu_drop_col.show()

+---------------+--------------+---------------+---------------+-----------------+-------------+
|collision_index|casualty_class|sex_of_casualty|age_of_casualty|casualty_severity|casualty_type|
+---------------+--------------+---------------+---------------+-----------------+-------------+
|  197905H200700|             1|              1|             20|                3|          109|
|  197952FKE0696|             2|              2|             18|                3|          109|
|  197997NC01901|             2|              1|             65|                3|          109|
|  1979316110883|             2|              1|             52|                3|          109|
|  197901F9MCD21|             2|              1|             87|                3|           11|
|  197901JARGE70|             1|              2|             19|                3|          104|
|  19794000B0984|             2|              1|             24|                3|          109|
|  1979557920388|             

In [10]:
df_casu_drop_col.schema

StructType([StructField('collision_index', StringType(), True), StructField('casualty_class', StringType(), True), StructField('sex_of_casualty', StringType(), True), StructField('age_of_casualty', StringType(), True), StructField('casualty_severity', StringType(), True), StructField('casualty_type', StringType(), True)])

In [11]:
df_casu_drop_col.count()

11974250

In [12]:
df_casualty_1979_2024 = df_casu_drop_col.select(
    F.col("collision_index"),
    F.col("casualty_class").cast(IntegerType()),
    F.col("sex_of_casualty").cast(IntegerType()),
    F.col("age_of_casualty").cast(IntegerType()),
    F.col("casualty_severity").cast(IntegerType()),
    F.col("casualty_type").cast(IntegerType()),
)

In [13]:
df_casualty_1979_2024.schema

StructType([StructField('collision_index', StringType(), True), StructField('casualty_class', IntegerType(), True), StructField('sex_of_casualty', IntegerType(), True), StructField('age_of_casualty', IntegerType(), True), StructField('casualty_severity', IntegerType(), True), StructField('casualty_type', IntegerType(), True)])

In [14]:
df_casualty_1979_2024.write.parquet("data/casualty_1979_2024", mode='overwrite')

In [15]:
df_collision = spark.read \
    .option("header", "true") \
    .csv('data/dft-road-casualty-statistics-collision-1979-latest-published-year.csv')

In [16]:
df_collision.head()

26/03/09 21:24:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Row(collision_index='2018450290294', collision_year='2018', collision_ref_no='450290294', location_easting_osgr='487906', location_northing_osgr='132932', longitude='-0.746206', latitude='51.08897', police_force='45', collision_severity='3', number_of_vehicles='2', number_of_casualties='2', date='15/04/2018', day_of_week='1', time='08:00', local_authority_district='517', local_authority_ons_district='E07000216', local_authority_highway='E10000030', local_authority_highway_current='E10000030', first_road_class='6', first_road_number='0', road_type='6', speed_limit='30', junction_detail_historic='0', junction_detail='0', junction_control='-1', second_road_class='0', second_road_number='-1', pedestrian_crossing_human_control_historic='0', pedestrian_crossing_physical_facilities_historic='0', pedestrian_crossing='0', light_conditions='1', weather_conditions='9', road_surface_conditions='1', special_conditions_at_site='0', carriageway_hazards_historic='0', carriageway_hazards='0', urban_or_

In [17]:
columns_collision = [
    'collision_index',
    'collision_severity', #
    'date',
    'day_of_week',
    'time', #
    'road_type',
    'speed_limit',
    'light_conditions',
    'weather_conditions',
    'road_surface_conditions',
    'urban_or_rural_area'
]

In [18]:
col_coll_to_drop = []
for col in df_collision.columns:
    if col not in columns_collision:
        col_coll_to_drop.append(col)

In [19]:
df_collision_drop_col = df_collision.drop(*col_coll_to_drop)

In [20]:
df_collision_drop_col.show()

+---------------+------------------+----------+-----------+-----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|collision_index|collision_severity|      date|day_of_week| time|road_type|speed_limit|light_conditions|weather_conditions|road_surface_conditions|urban_or_rural_area|
+---------------+------------------+----------+-----------+-----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|  2018450290294|                 3|15/04/2018|          1|08:00|        6|         30|               1|                 9|                      1|                  1|
|  2020230984311|                 2|21/09/2020|          2|23:00|        3|         70|               6|                 1|                      1|                  2|
|  1998929800391|                 3|29/01/1998|          5|11:00|        6|         30|               1|                 1|                      1|             

In [21]:
df_collision_drop_col.schema

StructType([StructField('collision_index', StringType(), True), StructField('collision_severity', StringType(), True), StructField('date', StringType(), True), StructField('day_of_week', StringType(), True), StructField('time', StringType(), True), StructField('road_type', StringType(), True), StructField('speed_limit', StringType(), True), StructField('light_conditions', StringType(), True), StructField('weather_conditions', StringType(), True), StructField('road_surface_conditions', StringType(), True), StructField('urban_or_rural_area', StringType(), True)])

In [22]:
df_collision_drop_col.count()

9015100

In [23]:
df_collision_1979_2024 = df_collision_drop_col.select(
    F.col("collision_index"),
    F.col("collision_severity").cast(IntegerType()),
    F.to_date(F.col("date"), "dd/MM/yyyy").alias("date"),
    F.col("day_of_week").cast(IntegerType()),
    F.split(F.col("time"), ":")[0].cast(IntegerType()).alias("time"),
    F.col("road_type").cast(IntegerType()),
    F.col("speed_limit").cast(IntegerType()),
    F.col("light_conditions").cast(IntegerType()),
    F.col("weather_conditions").cast(IntegerType()),
    F.col("road_surface_conditions").cast(IntegerType()),
    F.col("urban_or_rural_area").cast(IntegerType()),
)

In [24]:
df_collision_1979_2024.show()

+---------------+------------------+----------+-----------+----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|collision_index|collision_severity|      date|day_of_week|time|road_type|speed_limit|light_conditions|weather_conditions|road_surface_conditions|urban_or_rural_area|
+---------------+------------------+----------+-----------+----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|  2018450290294|                 3|2018-04-15|          1|   8|        6|         30|               1|                 9|                      1|                  1|
|  2020230984311|                 2|2020-09-21|          2|  23|        3|         70|               6|                 1|                      1|                  2|
|  1998929800391|                 3|1998-01-29|          5|  11|        6|         30|               1|                 1|                      1|                  2

In [25]:
df_collision_1979_2024.write.parquet("data/collision_1979_2024", mode='overwrite')

In [26]:
df_coll = spark.read.parquet("data/collision_1979_2024")
df_casu = spark.read.parquet("data/casualty_1979_2024")

In [27]:
df_coll.count()

9015100

In [28]:
df_casu.count()

11974250

In [29]:
df_1979_2024 = df_casu.join(df_coll, on="collision_index", how="inner")

In [30]:
df_1979_2024.show()

[Stage 28:>                                                         (0 + 1) / 1]

+---------------+--------------+---------------+---------------+-----------------+-------------+------------------+----------+-----------+----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|collision_index|casualty_class|sex_of_casualty|age_of_casualty|casualty_severity|casualty_type|collision_severity|      date|day_of_week|time|road_type|speed_limit|light_conditions|weather_conditions|road_surface_conditions|urban_or_rural_area|
+---------------+--------------+---------------+---------------+-----------------+-------------+------------------+----------+-----------+----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|  197901A1BGF95|             2|              2|             62|                2|          109|                 2|1979-01-01|          2|   1|        6|         30|               4|                 3|                      3|                 -1|
|  197901A1FDC88

In [32]:
df_1979_2024.count()

11974250

In [31]:
df_1979_2024.write.parquet('data/aggregation/1979-2024', mode='overwrite')

In [32]:
df_casualty_2025 = spark.read \
    .option("header", "true") \
    .csv('data/dft-road-casualty-statistics-casualty-provisional-2025.csv')

In [33]:
df_casu_2025_drop_col = df_casualty_2025.drop(*col_to_drop)

In [34]:
df_casu_2025_drop_col.show()

+---------------+--------------+---------------+---------------+-----------------+-------------+
|collision_index|casualty_class|sex_of_casualty|age_of_casualty|casualty_severity|casualty_type|
+---------------+--------------+---------------+---------------+-----------------+-------------+
|  2025010551784|             1|              1|             55|                3|            9|
|  2025010551786|             1|              1|             30|                3|            3|
|  2025010551792|             1|              1|             29|                3|            9|
|  2025010551794|             3|              1|             59|                2|            0|
|  2025010551794|             3|              1|             -1|                3|            0|
|  2025010551795|             1|              2|             25|                2|            1|
|  2025010551802|             1|              1|             25|                2|            1|
|  2025010551803|             

In [35]:
df_casu_2025_drop_col.count()

60991

In [36]:
df_casualty_2025 = df_casu_2025_drop_col.select(
    F.col("collision_index"),
    F.col("casualty_class").cast(IntegerType()),
    F.col("sex_of_casualty").cast(IntegerType()),
    F.col("age_of_casualty").cast(IntegerType()),
    F.col("casualty_severity").cast(IntegerType()),
    F.col("casualty_type").cast(IntegerType()),
)

In [37]:
df_casualty_2025.schema

StructType([StructField('collision_index', StringType(), True), StructField('casualty_class', IntegerType(), True), StructField('sex_of_casualty', IntegerType(), True), StructField('age_of_casualty', IntegerType(), True), StructField('casualty_severity', IntegerType(), True), StructField('casualty_type', IntegerType(), True)])

In [38]:
df_collision_2025 = spark.read \
    .option("header", "true") \
    .csv('data/dft-road-casualty-statistics-collision-provisional-2025.csv')

In [39]:
df_coll_2025_drop_col = df_collision_2025.drop(*col_coll_to_drop)

In [40]:
df_coll_2025_drop_col.show()

+---------------+------------------+----------+-----------+-----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|collision_index|collision_severity|      date|day_of_week| time|road_type|speed_limit|light_conditions|weather_conditions|road_surface_conditions|urban_or_rural_area|
+---------------+------------------+----------+-----------+-----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|  2025010551784|                 3|01/01/2025|          4|00:04|        6|         20|               4|                 1|                      1|                  1|
|  2025010551786|                 3|01/01/2025|          4|00:04|        6|         20|               4|                 1|                      1|                  1|
|  2025010551792|                 3|01/01/2025|          4|01:35|        6|         30|               4|                 4|                      1|             

In [41]:
df_coll_2025_drop_col.count()

48472

In [42]:
df_collision_2025 = df_coll_2025_drop_col.select(
    F.col("collision_index"),
    F.col("collision_severity").cast(IntegerType()),
    F.to_date(F.col("date"), "dd/MM/yyyy").alias("date"),
    F.col("day_of_week").cast(IntegerType()),
    F.split(F.col("time"), ":")[0].cast(IntegerType()).alias("time"),
    F.col("road_type").cast(IntegerType()),
    F.col("speed_limit").cast(IntegerType()),
    F.col("light_conditions").cast(IntegerType()),
    F.col("weather_conditions").cast(IntegerType()),
    F.col("road_surface_conditions").cast(IntegerType()),
    F.col("urban_or_rural_area").cast(IntegerType()),
)

In [43]:
df_collision_2025.schema

StructType([StructField('collision_index', StringType(), True), StructField('collision_severity', IntegerType(), True), StructField('date', DateType(), True), StructField('day_of_week', IntegerType(), True), StructField('time', IntegerType(), True), StructField('road_type', IntegerType(), True), StructField('speed_limit', IntegerType(), True), StructField('light_conditions', IntegerType(), True), StructField('weather_conditions', IntegerType(), True), StructField('road_surface_conditions', IntegerType(), True), StructField('urban_or_rural_area', IntegerType(), True)])

In [44]:
df_collision_2025.show()

+---------------+------------------+----------+-----------+----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|collision_index|collision_severity|      date|day_of_week|time|road_type|speed_limit|light_conditions|weather_conditions|road_surface_conditions|urban_or_rural_area|
+---------------+------------------+----------+-----------+----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|  2025010551784|                 3|2025-01-01|          4|   0|        6|         20|               4|                 1|                      1|                  1|
|  2025010551786|                 3|2025-01-01|          4|   0|        6|         20|               4|                 1|                      1|                  1|
|  2025010551792|                 3|2025-01-01|          4|   1|        6|         30|               4|                 4|                      1|                  1

In [45]:
df_2025 = df_casualty_2025.join(df_collision_2025, on="collision_index", how="inner")

In [46]:
df_2025.count()

60991

In [47]:
df_2025.write.parquet('data/aggregation/2025', mode='overwrite')

In [48]:
df_aggregation = df_1979_2024.unionByName(df_2025)

In [ ]:
df_aggregation.write.parquet('data/aggregation/1979-2025', mode = 'overwrite')

[Stage 56:>                                                        (0 + 4) / 13]